# Day 6 — Logistic regression from scratch: sigmoid, cross-entropy, and a real gradient bug

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import RobustScaler, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import log_loss

## Setup — reuse Day 2's Titanic cleaning

Same feature set as Day 2 (`pclass`, `fare`, `who`, `family_size`), but kept **unscaled** for now — that's the point. `fare` runs 0 to 512; the others are single digits. We'll feel the consequence of that mismatch in Step 4.

In [ ]:
df = sns.load_dataset("titanic")
df["sex"] = df["sex"].map({"male": 0, "female": 1})
df["embarked"] = df["embarked"].map({"C": 0, "Q": 1, "S": 2})
df["family_size"] = df["sibsp"] + df["parch"] + 1
df.drop(
    columns=[
        "class",
        "embark_town",
        "alive",
        "alone",
        "sibsp",
        "parch",
        "deck",
        "adult_male",
    ],
    inplace=True,
)

median_age = df["age"].median()
df["age"] = df["age"].fillna(median_age)
mode_embarked = df["embarked"].mode()[0]
df["embarked"] = df["embarked"].fillna(mode_embarked)
df["embarked"] = df["embarked"].astype(int)
df["who"] = df["who"].map({"man": 0, "woman": 1, "child": 2})
df.drop(["sex", "age", "embarked"], axis=1, inplace=True)

x_train, x_test, y_train, y_test = train_test_split(
    df.drop("survived", axis=1), df["survived"], test_size=0.2, random_state=42
)

# raw numpy views — this is what the from-scratch functions below operate on
X_train = x_train.values.astype(float)
X_test = x_test.values.astype(float)
y_train_v = y_train.values.astype(float)
y_test_v = y_test.values.astype(float)

x_train.describe()

## Step 1 — sigmoid sanity checks

`sigmoid(z) = 1 / (1 + e^-z)` maps any raw score to a probability. Check the three landmark points: `z=0` should sit exactly at the midpoint, and large positive/negative `z` should saturate toward 1/0.

In [ ]:
def sigmoid(z):
    return 1 / (1 + np.exp(-z))


print("sigmoid(0):  ", sigmoid(0))
print("sigmoid(10): ", sigmoid(10))
print("sigmoid(-10):", sigmoid(-10))

## Step 2 — cross-entropy, verified against a real fitted model

Before trusting `cross_entropy` inside a gradient loop, check it against `sklearn.metrics.log_loss` on real predicted probabilities. Fit a quick `LogisticRegression` (Day 2 style — `RobustScaler` on `fare` only, since fare is right-skewed with genuine outliers) and compare.

In [ ]:
def cross_entropy(y_true, y_pred, eps=1e-15):
    y_pred = np.clip(y_pred, eps, 1 - eps)
    return -np.mean(y_true * np.log(y_pred) + (1 - y_true) * np.log(1 - y_pred))


scaler = RobustScaler()
x_train_scaled = x_train.copy()
x_test_scaled = x_test.copy()
x_train_scaled[["fare"]] = scaler.fit_transform(x_train[["fare"]])
x_test_scaled[["fare"]] = scaler.transform(x_test[["fare"]])

model = LogisticRegression()
model.fit(x_train_scaled, y_train)

probs_test = model.predict_proba(x_test_scaled)[:, 1]
manual_ce = cross_entropy(y_test_v, probs_test)
sk_ll = log_loss(y_test_v, probs_test)

print("manual cross-entropy:", manual_ce)
print("sklearn log_loss:    ", sk_ll)
print("match:", np.isclose(manual_ce, sk_ll))

## Step 3 — implement the gradient

For logistic regression + cross-entropy, the chain rule collapses to a clean form: `dw = (1/n) X^T (p - y)`, `db = mean(p - y)`. `error = p - y` is how wrong each prediction is; `dw` is just that error, averaged per feature.

In [ ]:
def gradients_logreg(w, b, X, y, lam=0.0):
    n = len(y)
    p = sigmoid(X @ w + b)
    error = p - y
    dw = (1 / n) * (X.T @ error) + (lam / n) * w
    db = (1 / n) * np.sum(error)
    return dw, db


def cost_logreg(w, b, X, y, eps=1e-15, lam=0.0):
    p = sigmoid(X @ w + b)
    return cross_entropy(y, p, eps=eps) + (lam / (2 * len(y))) * np.sum(w**2)

## Step 4 — gradient check, and it fails

Same numerical-vs-analytical check as Day 1/Day 5's from-scratch models, run on the **unscaled** features (`pclass`, `fare`, `who`, `family_size` straight from the train split, no scaling).

In [ ]:
rng = np.random.default_rng(42)
w_test = rng.normal(0, 0.1, size=X_train.shape[1])  # arbitrary small random weights
b_test = 0.0
eps = 1e-4  # tiny nudge for the finite-difference approximation

dw_analytical, db_analytical = gradients_logreg(w_test, b_test, X_train, y_train_v)

dw_numerical = np.zeros_like(w_test)
for i in range(len(w_test)):
    w_plus = w_test.copy()
    w_plus[i] += eps
    w_minus = w_test.copy()
    w_minus[i] -= eps
    dw_numerical[i] = (
        cost_logreg(w_plus, b_test, X_train, y_train_v)
        - cost_logreg(w_minus, b_test, X_train, y_train_v)
    ) / (2 * eps)

print("dw analytical:", dw_analytical)
print("dw numerical: ", dw_numerical)
print("max abs diff: ", np.max(np.abs(dw_analytical - dw_numerical)))

## Step 5 — diagnose it

The `fare` gradient is the outlier — off by ~2, stable across `eps` from `1e-4` down to `1e-7` (checked separately), so it's not finite-difference noise. Check the raw scores `z = Xw + b` the model is actually producing.

In [ ]:
z = X_train @ w_test + b_test
print("z min:", z.min(), " z max:", z.max())
print("sigmoid(z.min()):", sigmoid(z.min()))

# fare reaches 512 unscaled — even a small random weight on it dominates z and
# pushes some samples to z ~ -53. sigmoid(-53) sits at the edge of float64
# precision: the analytical gradient works with that raw probability directly,
# while the numerical check computes cost(w+eps) - cost(w-eps) — a tiny
# difference between two numbers both already squeezed near a precision floor.
# Subtracting them loses precision (catastrophic cancellation) — a real
# numerical bug, not a wrong formula.

## Step 6 — fix it, verified

Scale the numeric columns — fit on train only, same leakage discipline as every day this week. Then rerun the exact same gradient check.

In [ ]:
std_scaler = StandardScaler()
X_train_std = std_scaler.fit_transform(X_train)

z_std = X_train_std @ w_test + b_test
print("z range (scaled):", z_std.min(), "to", z_std.max())

dw_analytical2, db_analytical2 = gradients_logreg(
    w_test, b_test, X_train_std, y_train_v
)

dw_numerical2 = np.zeros_like(w_test)
for i in range(len(w_test)):
    w_plus = w_test.copy()
    w_plus[i] += eps
    w_minus = w_test.copy()
    w_minus[i] -= eps
    dw_numerical2[i] = (
        cost_logreg(w_plus, b_test, X_train_std, y_train_v)
        - cost_logreg(w_minus, b_test, X_train_std, y_train_v)
    ) / (2 * eps)

print("dw analytical:", dw_analytical2)
print("dw numerical: ", dw_numerical2)
print("max abs diff: ", np.max(np.abs(dw_analytical2 - dw_numerical2)))

# Step 7 - The tranining loop

In [ ]:
n_features = X_train_std.shape[1]
w = np.zeros(n_features)
b = 0.0
lr = 0.5
n_iters = 2000

cost_history = []
for i in range(n_iters):
    dw, db = gradients_logreg(w, b, X_train_std, y_train_v)
    w -= lr * dw
    b -= lr * db
    if i % 50 == 0 or i == n_iters - 1:
        cost_history.append((i, cost_logreg(w, b, X_train_std, y_train_v)))

print("final w:", w)
print("final b:", b)
print("cost at iter 0, 50, 200, 500, last:")
for i, c in cost_history:
    if i in (0, 50, 200, 500, n_iters - 1):
        print(f"  iter {i}: {c:.4f}")

## Step 8 — learning rate exploration

Retrain from scratch (fresh `w=0, b=0`) at a few different `lr` values, for the same `n_iters`. Compare final train cost and test accuracy to see which rates have actually converged vs. which are still crawling toward the minimum.

In [ ]:
X_test_std = std_scaler.transform(X_test)


def train_logreg(lr, n_iters=2000, lam=0.0):
    w = np.zeros(n_features)
    b = 0.0
    for _ in range(n_iters):
        dw, db = gradients_logreg(w, b, X_train_std, y_train_v, lam=lam)
        w -= lr * dw
        b -= lr * db
    return w, b


for lr_try in [0.01, 0.1, 0.5, 1.0]:
    w_lr, b_lr = train_logreg(lr_try)
    train_cost = cost_logreg(w_lr, b_lr, X_train_std, y_train_v)
    test_preds = (sigmoid(X_test_std @ w_lr + b_lr) >= 0.5).astype(int)
    test_acc = (test_preds == y_test_v).mean()
    print(
        f"lr={lr_try:<5} final train cost={train_cost:.4f}  test accuracy={test_acc:.4f}"
    )

## Step 9 — the headline result

Use the `w, b` already trained in Step 7 at `lr=0.5` (2000 iterations). Check the convergence curve, then evaluate on the test set and compare accuracy + confusion matrix against a fresh `sklearn.LogisticRegression` fit on the identical standardized features.

In [ ]:
from sklearn.metrics import confusion_matrix, accuracy_score

# print("convergence curve (iter, cost):")
# for i, c in cost_history:
#     print(f"  {i}: {c:.4f}")

test_preds = (sigmoid(X_test_std @ w + b) >= 0.5).astype(int)
scratch_acc = accuracy_score(y_test_v, test_preds)
scratch_cm = confusion_matrix(y_test_v, test_preds)
print("\nfrom-scratch test accuracy:", scratch_acc)
print("from-scratch confusion matrix:\n", scratch_cm)

sk_baseline = LogisticRegression()
sk_baseline.fit(X_train_std, y_train_v)
sk_preds = sk_baseline.predict(X_test_std)
sk_acc = accuracy_score(y_test_v, sk_preds)
sk_cm = confusion_matrix(y_test_v, sk_preds)
print("\nsklearn test accuracy:", sk_acc)
print("sklearn confusion matrix:\n", sk_cm)

print("\nexact match:", scratch_acc == sk_acc and (scratch_cm == sk_cm).all())

## Step 10 — matching predictions, mismatched coefficients

Fit `sklearn.LogisticRegression(penalty=None)` on the identical standardized features — unregularized, so it's a fair fight against the from-scratch model. Predictions matched exactly in Step 9; check whether the learned coefficients did too.

In [ ]:
sk_unreg = LogisticRegression(penalty=None, max_iter=2000)
sk_unreg.fit(X_train_std, y_train_v)

print("from-scratch w:", w)
print("sklearn unreg w:", sk_unreg.coef_[0])
print("max abs coefficient diff:", np.max(np.abs(w - sk_unreg.coef_[0])))

## Step 11 — add L2 regularization

`gradients_logreg`/`cost_logreg` now take `lam` (the regularization strength, `λ`), defaulting to `0.0` so every earlier cell is untouched. The penalty only applies to `w`, never `b` — the bias isn't a "feature weight," it's just an offset, so there's nothing to shrink it toward.

`lam=1.0` here isn't arbitrary: it's chosen to match sklearn's *default* `LogisticRegression()` (`C=1.0`), since sklearn's `C` is an inverse regularization strength (`lam ≈ 1/C`). That makes `sk_baseline` from Step 9 — already fit with default L2 — the natural thing to compare against.

In [ ]:
w_l2, b_l2 = train_logreg(lr=0.5, n_iters=2000, lam=1.0)

print("unregularized w (Step 7): ", w)
print("L2-regularized w (lam=1):", w_l2)
print("sklearn default (L2) w:  ", sk_baseline.coef_[0])
print()
print(
    "max abs diff vs sklearn default L2:", np.max(np.abs(w_l2 - sk_baseline.coef_[0]))
)
print(
    "max abs diff vs sklearn unreg (Step 10):", np.max(np.abs(w_l2 - sk_unreg.coef_[0]))
)

test_preds_l2 = (sigmoid(X_test_std @ w_l2 + b_l2) >= 0.5).astype(int)
print("\nL2 test accuracy:", accuracy_score(y_test_v, test_preds_l2))